参照NeXtMD模型的fig2进行模仿

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from collections import Counter
import os


In [5]:
# ==========================================
# 【字体设置】全局使用 Arial
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.sans-serif'] = ['Arial']

# 【颜色设置】
COLOR_POS = "#FF7F00"  # 阳性 - 深橙色
COLOR_NEG = "#1F78B4"  # 阴性 - 深蓝色

# 【路径设置】
# 输入文件路径 (根据您的截图确认的路径)
# 注意：如果路径中有空格，请确保 r"..." 内部字符串与实际文件夹名一致
csv_file_path = r"E:\LLM+XWT\XWT数据\Umami-BERT-UMP789\1_1-data.csv"

# 输出保存路径 (您指定的新路径)
output_dir = r"E:\LLM+XWT\UMP789描述"

# 自动创建保存文件夹
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已创建输出目录: {output_dir}")

# ==========================================
# 2. 数据读取 (这里进行了修改)
# ==========================================
print(f"正在读取文件: {csv_file_path} ...")

if os.path.exists(csv_file_path):
    # 读取 CSV
    df = pd.read_csv(csv_file_path)
    
    # --- 关键点：请核对您的 CSV 列名 ---
    # 假设您的 CSV 中序列列名为 'Sequence'，标签列名为 'Label' (1为阳性, 0为阴性)
    # 如果报错 "KeyError"，请将下面的 'Sequence' 和 'Label' 改为您的实际列名
    try:
        # 提取阳性序列 (Label == 1)
        total_pos_seqs = df[df['label'] == 1]['seq'].tolist()
        # 提取阴性序列 (Label == 0)
        total_neg_seqs = df[df['label'] == 0]['seq'].tolist()
        
        # 转大写并去空格 (清洗数据)
        total_pos_seqs = [s.strip().upper() for s in total_pos_seqs if isinstance(s, str)]
        total_neg_seqs = [s.strip().upper() for s in total_neg_seqs if isinstance(s, str)]
        
        print(f"读取成功!")
        print(f"阳性样本数 (Positive): {len(total_pos_seqs)}")
        print(f"阴性样本数 (Negative): {len(total_neg_seqs)}")
        
    except KeyError as e:
        print(f"\n!!! 错误：找不到列名 {e}。")
        print(f"您的CSV列名是: {list(df.columns)}")
        print("请修改代码中的 df['Label'] 或 df['Sequence'] 为实际列名。\n")
        # 为了防止后面报错，给个空列表
        total_pos_seqs, total_neg_seqs = [], []
else:
    print(f"错误：找不到文件 {csv_file_path}")
    total_pos_seqs, total_neg_seqs = [], []

# 理化性质数据 (保持不变)
aa_properties = {
    'A': [0.62, 0.5, 0.0, 0.05], 'R': [0.0, 1.0, 1.0, 0.65], 'N': [0.11, 0.5, 1.0, 0.35],
    'D': [0.11, 0.0, 1.0, 0.36], 'C': [0.29, 0.5, 0.0, 0.27], 'Q': [0.11, 0.5, 1.0, 0.44],
    'E': [0.11, 0.0, 1.0, 0.45], 'G': [0.48, 0.5, 0.0, 0.00], 'H': [0.17, 0.8, 1.0, 0.52],
    'I': [1.00, 0.5, 0.0, 0.36], 'L': [0.94, 0.5, 0.0, 0.36], 'K': [0.00, 1.0, 1.0, 0.44],
    'M': [0.74, 0.5, 0.0, 0.47], 'F': [0.88, 0.5, 0.0, 0.58], 'P': [0.32, 0.5, 0.0, 0.20],
    'S': [0.26, 0.5, 1.0, 0.14], 'T': [0.34, 0.5, 1.0, 0.25], 'W': [0.81, 0.5, 0.2, 0.92],
    'Y': [0.63, 0.5, 0.5, 0.68], 'V': [0.91, 0.5, 0.0, 0.22]
}
aa_order = sorted(list(aa_properties.keys()))
prop_names = ['Hydrophobicity', 'Net Charge', 'Polarity', 'Molecular Weight']

正在读取文件: E:\LLM+XWT\XWT数据\Umami-BERT-UMP789\1_1-data.csv ...
读取成功!
阳性样本数 (Positive): 307
阴性样本数 (Negative): 307


In [6]:
print("-" * 30)
print(f"【全数据集统计】")
print(f"Total Positive: {len(total_pos_seqs)}")
print(f"Total Negative: {len(total_neg_seqs)}")
print(f"Total Sequences: {len(total_pos_seqs) + len(total_neg_seqs)}")
print("-" * 30)

------------------------------
【全数据集统计】
Total Positive: 307
Total Negative: 307
Total Sequences: 614
------------------------------


In [7]:

# ==========================================
# 3. 绘图部分 (分开绘制并保存)
# ==========================================

# --- 图 1: 序列长度分布 (Sequence Length Distribution) ---
plt.figure(figsize=(10, 8)) # ### 这里调整画布大小 (宽, 高)
ax1 = plt.gca()

pos_lens = Counter([len(s) for s in total_pos_seqs])
neg_lens = Counter([len(s) for s in total_neg_seqs])
all_lens = sorted(set(pos_lens.keys()) | set(neg_lens.keys()))
pos_counts = [pos_lens.get(l, 0) for l in all_lens]
neg_counts = [-neg_lens.get(l, 0) for l in all_lens]

ax1.barh(all_lens, neg_counts, color=COLOR_NEG, label='Negative', edgecolor='white', height=0.8)
ax1.barh(all_lens, pos_counts, color=COLOR_POS, label='Positive', edgecolor='white', height=0.8)

# ### 坐标轴标签设置：fontsize调整字号，fontweight='bold'加粗
ax1.set_xlabel('Number of Peptides', fontsize=23, fontweight='bold') 
ax1.set_ylabel('Sequence Length (AA)', fontsize=23, fontweight='bold')
ax1.axvline(0, color='black', linewidth=0.8)

# ### 刻度文字设置：labelsize调整字号
ax1.tick_params(axis='both', labelsize=22) 
ticks = ax1.get_xticks()
ax1.set_xticks(ticks)
ax1.set_xticklabels([str(abs(int(x))) for x in ticks], fontweight='bold')
# 如果您希望 Y轴的刻度（0, 5, 10...）也同步加粗，请加上这一行：
plt.setp(ax1.get_yticklabels(), fontweight='bold')

# ### 图例设置：fontsize调整字号，loc调整位置
ax1.legend(fontsize=23, loc='upper right', ) 

# 保存
save_path = os.path.join(output_dir, "Fig_Sequence_Length.png")
plt.savefig(save_path, dpi=500, bbox_inches='tight') # ### dpi=500 设置分辨率
plt.close()
print(f"已保存: {save_path}")



已保存: E:\LLM+XWT\UMP789描述\Fig_Sequence_Length.png


In [8]:
# --- 图 2: 氨基酸组分 (更新字号) ---
plt.figure(figsize=(12, 8)) 
ax2 = plt.gca()

def count_aa(seq_list):
    cnt = Counter()
    for s in seq_list:
        for aa in s:
            if aa in aa_order: cnt[aa] += 1
    return cnt

pos_aa_cnt = count_aa(total_pos_seqs)
neg_aa_cnt = count_aa(total_neg_seqs)

df_aa = pd.DataFrame({
    'Amino Acid': aa_order * 2,
    'Count': [pos_aa_cnt[aa] for aa in aa_order] + [neg_aa_cnt[aa] for aa in aa_order],
    'Type': ['Positive'] * 20 + ['Negative'] * 20
})

sns.barplot(x='Amino Acid', y='Count', hue='Type', data=df_aa, 
            palette={'Positive': COLOR_POS, 'Negative': COLOR_NEG}, 
            ax=ax2, edgecolor='white', width=0.8)

# ### 设置坐标轴标签 (fontsize=23, bold)
ax2.set_xlabel('Amino Acid', fontsize=23, fontweight='bold')
ax2.set_ylabel('Total Count', fontsize=23, fontweight='bold')

# ### 设置刻度 (labelsize=22, bold)
ax2.tick_params(axis='both', labelsize=22)
plt.setp(ax2.get_xticklabels(), fontweight='bold')
plt.setp(ax2.get_yticklabels(), fontweight='bold')

# ### 设置图例 (fontsize=23)
ax2.legend(title=None, fontsize=23)

save_path = os.path.join(output_dir, "Fig_AA_Composition.png")
plt.savefig(save_path, dpi=500, bbox_inches='tight')
plt.close()
print(f"已保存: {save_path}")


已保存: E:\LLM+XWT\UMP789描述\Fig_AA_Composition.png


In [9]:
# --- 准备热图数据 ---
def create_property_matrix(seqs, aa_cnt_dict):
    total_aa = sum(aa_cnt_dict.values())
    matrix = np.zeros((len(prop_names), 20))
    for i, prop in enumerate(prop_names):
        for j, aa in enumerate(aa_order):
            freq = aa_cnt_dict[aa] / total_aa if total_aa > 0 else 0
            raw_val = aa_properties[aa][i]
            matrix[i, j] = freq * raw_val
    if np.max(matrix) > 0: matrix = matrix / np.max(matrix)
    return pd.DataFrame(matrix, index=prop_names, columns=aa_order)

df_heatmap_pos = create_property_matrix(total_pos_seqs, pos_aa_cnt)
df_heatmap_neg = create_property_matrix(total_neg_seqs, neg_aa_cnt)

# --- 图 3: 阳性热图 (更新字号) ---
plt.figure(figsize=(12, 6)) 
ax3 = plt.gca()

sns.heatmap(df_heatmap_pos, cmap='Oranges', linewidths=.5, ax=ax3, 
            cbar_kws={'label': 'Normalized Contribution'})

# ### 设置色条 (Colorbar)
cbar = ax3.collections[0].colorbar
# 色条标题: 23号加粗
cbar.set_label('Normalized Contribution', fontsize=23, fontweight='bold')
# 色条刻度: 22号加粗
cbar.ax.tick_params(labelsize=22)
plt.setp(cbar.ax.get_yticklabels(), fontweight='bold')

# ### 设置坐标轴标签
ax3.set_xlabel('Residue', fontsize=23, fontweight='bold')
ax3.set_ylabel('')

# ### 设置刻度
ax3.tick_params(axis='x', labelsize=22) 
ax3.tick_params(axis='y', labelsize=22, rotation=0)
plt.setp(ax3.get_xticklabels(), fontweight='bold')
plt.setp(ax3.get_yticklabels(), fontweight='bold')

save_path = os.path.join(output_dir, "Fig_Heatmap_Positive.png")
plt.savefig(save_path, dpi=500, bbox_inches='tight')
plt.close()
print(f"已保存: {save_path}")

已保存: E:\LLM+XWT\UMP789描述\Fig_Heatmap_Positive.png


In [10]:
# --- 图 4: 阴性热图 (更新字号) ---
plt.figure(figsize=(12, 6))
ax4 = plt.gca()

sns.heatmap(df_heatmap_neg, cmap='Blues', linewidths=.5, ax=ax4, 
            cbar_kws={'label': 'Normalized Contribution'})

# ### 设置色条
cbar = ax4.collections[0].colorbar
cbar.set_label('Normalized Contribution', fontsize=23, fontweight='bold')
cbar.ax.tick_params(labelsize=22)
plt.setp(cbar.ax.get_yticklabels(), fontweight='bold')

# ### 设置坐标轴
ax4.set_xlabel('Residue', fontsize=23, fontweight='bold')
ax4.set_ylabel('')

# ### 设置刻度
ax4.tick_params(axis='x', labelsize=22)
ax4.tick_params(axis='y', labelsize=22, rotation=0)
plt.setp(ax4.get_xticklabels(), fontweight='bold')
plt.setp(ax4.get_yticklabels(), fontweight='bold')

save_path = os.path.join(output_dir, "Fig_Heatmap_Negative.png")
plt.savefig(save_path, dpi=500, bbox_inches='tight')
plt.close()
print(f"已保存: {save_path}")

print("-" * 30)
print(f"所有图片已按新字号保存至: {output_dir}")

已保存: E:\LLM+XWT\UMP789描述\Fig_Heatmap_Negative.png
------------------------------
所有图片已按新字号保存至: E:\LLM+XWT\UMP789描述
